In [ ]:
# Imports
import pandas as pd
import unicodedata

In [ ]:
# Geolocation paths
geoloc_base_path        = "data/campus_trace_geolocation_map_base.csv"
puclients_ipinfo_path   = "data/puclients_geolocation.csv"
external_ip_map_path    = "data/campus_trace_external_ip_map.csv"
geoloc_path_output      = "data/campus_trace_geolocation_map.csv"
geoloc_path_output_anon = "data/campus_trace_geolocation_map_anon.csv"
countries_geoloc_path   = "data/countries_geolocation.csv"
countries_naturalearth_path = "data/countries_naturalearth.csv"

In [ ]:
# Process base geolocation CSV
df_geo_anon = pd.read_csv(geoloc_base_path, low_memory=False)

# Convert latitude and longitude to numeric, coercing errors to NaN
df_geo_anon['Latitude']  = pd.to_numeric(df_geo_anon['Latitude'], errors='coerce')
df_geo_anon['Longitude'] = pd.to_numeric(df_geo_anon['Longitude'], errors='coerce')

# Drop rows where latitude or longitude is NaN
df_geo_anon = df_geo_anon.dropna(subset=['Latitude', 'Longitude'])

df_geo_anon = df_geo_anon.rename(columns={'External_IP_ID': 'Destination_IP_Anon_Old'})
print("df_geo_anon row count:", df_geo_anon.shape[0])

In [ ]:
df_geo_anon.head()

In [ ]:
# Load external IP to anonymous IP map
df_external_ip_map = pd.read_csv(external_ip_map_path, low_memory=False)
df_external_ip_map = df_external_ip_map.rename(columns={
    'External_IP_ID': 'Destination_IP_Anon_Old',
    'External_IP': 'Destination_IP'})

# Extract the /24 prefix by keeping only the first three octets
df_external_ip_map['Destination_Prefix'] = df_external_ip_map['Destination_IP'].apply(
    lambda ip: '.'.join(ip.split('.')[:3]) + '.0')

print("df_external_ip_map row count:", df_external_ip_map.shape[0])

In [ ]:
df_external_ip_map.head()

In [ ]:
# Combine anonymous IPs and real IPs (still anonymized, but in a prefix-preserved manner)
df_geo_base = pd.merge(df_external_ip_map, df_geo_anon, on='Destination_IP_Anon_Old', how='inner')

print("df_geo_base row count:", df_geo_base.shape[0])

In [ ]:
df_geo_base.head()

In [ ]:
# Process IPinfo-based more recent (and therefore potentially more accurate) geolocation CSV
df_geo_ipinfo = pd.read_csv(puclients_ipinfo_path, low_memory=False)

# Convert latitude and longitude to numeric, coercing errors to NaN
df_geo_ipinfo['Latitude']  = pd.to_numeric(df_geo_ipinfo['Latitude'], errors='coerce')
df_geo_ipinfo['Longitude'] = pd.to_numeric(df_geo_ipinfo['Longitude'], errors='coerce')

# Drop rows where latitude or longitude is NaN
df_geo_ipinfo = df_geo_ipinfo.dropna(subset=['Latitude', 'Longitude'])

# Retain only relevant columns
df_geo_ipinfo = df_geo_ipinfo[['Destination_Prefix', 'Continent', 'Country', 'Latitude', 'Longitude']]

print("df_geo_ipinfo row count:", df_geo_ipinfo.shape[0])

In [ ]:
df_geo_ipinfo.head()

In [ ]:
# Count overlap between base data and IPinfo data
matched_rows_count = df_geo_base['Destination_Prefix'].isin(df_geo_ipinfo['Destination_Prefix']).sum()
print(f"No. of matched rows: {matched_rows_count} ({round(matched_rows_count*100/df_geo_base.shape[0], 1)}%)")

In [ ]:
# Perform a left merge
df_geo = df_geo_base.merge(df_geo_ipinfo, on='Destination_Prefix', how='left', suffixes=('_A', '_B'))

# Find the common columns
common_columns = sorted(set(df_geo_base.columns) & set(df_geo_ipinfo.columns) - {'Destination_Prefix'})

# Replace values in common columns where there's a match
for col in common_columns:
    df_geo[col] = df_geo[col + '_B'].combine_first(df_geo[col + '_A'])

# Drop extra columns from B (the suffixed ones)
df_geo.drop(columns=[col + '_A' for col in common_columns] + [col + '_B' for col in common_columns], inplace=True)

df_geo.head()

In [ ]:
df_geo_without_anon = df_geo.copy()

# Unique IPs
unique_ips = sorted(df_geo_without_anon['Destination_IP'].unique().tolist())

# IP mapping
ip_mapping = {ip: f'ip{index+1}' for index, ip in enumerate(unique_ips)}

# Add the mapped prefix column
df_geo_without_anon['Destination_IP_Anon'] = df_geo_without_anon['Destination_IP'].map(ip_mapping)

# Create unique prefix to anon prefix mapping
unique_prefixes = sorted(df_geo_without_anon['Destination_Prefix'].unique().tolist())

# Create a mapping where each prefix gets a unique "p" number
prefix_mapping = {prefix: f'p{index+1}' for index, prefix in enumerate(unique_prefixes)}

# Add the mapped prefix column
df_geo_without_anon['Destination_Prefix_Anon'] = df_geo_without_anon['Destination_Prefix'].map(prefix_mapping)

# Rejig the order of columns
df_geo_without_anon = df_geo_without_anon[ [
    'Destination_Prefix', 'Destination_Prefix_Anon', 'Destination_IP', 'Destination_IP_Anon', 'Destination_IP_Anon_Old',
    'Continent', 'Country', 'Latitude', 'Longitude'] ]

# Remove accents from country names
df_geo_without_anon['Country'] = df_geo_without_anon.Country.apply(lambda c: unicodedata.normalize('NFKD', c).encode('ascii', 'ignore').decode('utf-8'))
size_before = df_geo_without_anon.shape[0]

# Remove countries not in Natural Earth dataset
countries_to_remove = ["Bouvet Island", "French Guiana", "Guadeloupe", "Martinique", "Reunion"]
df_geo_without_anon = df_geo_without_anon[~df_geo_without_anon['Country'].isin(countries_to_remove)]

# Replace values in the 'Name' column
country_names_map = {
    "Aland Islands": "Aland",
    "Bahamas": "The Bahamas",
    "Congo Republic": "Republic of the Congo",
    "Czech Republic": "Czechia",
    "DR Congo": "Democratic Republic of the Congo",
    "Eswatini": "eSwatini",
    "Hong Kong": "Hong Kong S.A.R.",
    "Macao": "Macao S.A.R",
    "Serbia": "Republic of Serbia",
    "St Kitts and Nevis": "Saint Kitts and Nevis",
    "St Vincent and Grenadines": "Saint Vincent and the Grenadines",
    "Tanzania": "United Republic of Tanzania",
    "The Netherlands": "Netherlands",
    "Timor-Leste": "East Timor",
    "Turkiye": "Turkey",
    "U.S. Virgin Islands": "United States Virgin Islands",
    "United States": "United States of America"
}
df_geo_without_anon['Country'] = df_geo_without_anon['Country'].replace(country_names_map)

print(f"df_geo_without_anon after removing countries not in Natural Earth dataset: {df_geo_without_anon.shape[0]} out of {size_before}"
      + f" ({round(df_geo_without_anon.shape[0]*100/size_before, 2)}%)")

In [ ]:
df_geo_without_anon.head()

In [ ]:
# Save all the geolocation data to a single CSV file
df_geo_without_anon.to_csv(geoloc_path_output, index=False)

## List of all countries in data

In [ ]:
df_countries_geoloc = pd.DataFrame({'Country': sorted(df_geo['Country'].unique().tolist())})
df_countries_geoloc.head()

In [ ]:
df_countries_geoloc.to_csv(countries_geoloc_path, index=False)

## Only anonymized IPs and prefixes

In [ ]:
# Add the mapped prefix column
df_geo_only_anon = df_geo_without_anon[ [
    'Destination_Prefix_Anon', 'Destination_IP_Anon',
    'Continent', 'Country', 'Latitude', 'Longitude'] ]

df_geo_only_anon.head()

In [ ]:
# Save all the anonymous geolocation data to a single CSV file
df_geo_anon.to_csv(geoloc_path_output_anon, index=False)